In [1]:
import os
import yaml
import numpy as np
import pandas as pd
from tqdm import tqdm

In [3]:
# Load konfigurasi dari root folder
with open("../config/config.yaml", "r") as file:
    config = yaml.safe_load(file)

RAW_DIR = "../data/raw/"
PROCESSED_DIR = "../data/processed/"

print("Konfigurasi berhasil!")

Konfigurasi berhasil!


In [ ]:
#Pipeline aplication_train

def pipeline_application_train(file_path):
    df = pd.read_csv(file_path)
    
    # Adjust anomali DAYS_EMPLOYED (1000 tahun) menjadi NaN secara otomatis
    df['DAYS_EMPLOYED_ANOM'] = df['DAYS_EMPLOYED'] == 365243
    df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
    
    df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365.25
    df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'] / -365.25
    df['CREDIT_TO_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
    df['ANNUITY_TO_INCOME_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']
    
    print(f"Dimensi data utama: {df.shape}")
    return df

df_apps = pipeline_application_train(os.path.join(RAW_DIR, "application_train.csv"))

Dimensi data utama: (307511, 127)


/var/folders/m4/x0713fb11qdbv6bg4_r6t6t00000gn/T/ipykernel_7792/1120264303.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['DAYS_EMPLOYED_ANOM'] = df['DAYS_EMPLOYED'] == 365243
/var/folders/m4/x0713fb11qdbv6bg4_r6t6t00000gn/T/ipykernel_7792/1120264303.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365.25
/var/folders/m4/x0713fb11qdbv6bg4_r6t6t00000gn/T/ipykernel_7792/1120264303.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame

In [9]:
#bureau

def pipeline_bureau(file_path):
    bureau = pd.read_csv(file_path)
    
    # Lakukan agregasi numerik (Hitung rata-rata, nilai maksimum, dan jumlah pinjaman lama)
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({
        'DAYS_CREDIT': ['min', 'max', 'mean'],
        'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum']
    })
    
    bureau_agg.columns = ['_'.join(col).upper() for col in bureau_agg.columns]
    bureau_agg.reset_index(inplace=True)
    
    # Add fitur total pinjaman aktif masa lalu
    bureau_agg.rename(columns={'DAYS_CREDIT_MIN': 'BUREAU_DAYS_CREDIT_MIN'}, inplace=True)
    
    print(f"Dimensi data bureau: {bureau_agg.shape}")
    return bureau_agg

# Jalankan pipa bureau
df_bureau = pipeline_bureau(os.path.join(RAW_DIR, "bureau.csv"))

Dimensi data bureau: (305811, 10)


In [10]:
#Merging tables yang ada

df_final = df_apps.merge(df_bureau, on='SK_ID_CURR', how='left')
df_final = df_final.copy()

os.makedirs(PROCESSED_DIR, exist_ok=True)

#Simpan data bersih hasil pabrikasi otomatis ke data/processed/
output_path = os.path.join(PROCESSED_DIR, "application_train_clean.csv")
df_final.to_csv(output_path, index=False)

print(f"File bersih tersimpan di: {output_path}")
print(f"Dimensi akhir dataset siap latih ML: {df_final.shape}")

File bersih tersimpan di: ../data/processed/application_train_clean.csv
Dimensi akhir dataset siap latih ML: (307511, 136)


In [ ]:

#git add notebooks/2-data_pipeline_and_aggregation.ipynb
#git add data/processed/
#git commit -m "Feature: Complete Data Pipeline and Bureau table aggregation. Exported 136 clean features to processed folder"
#git push origin main